In [ ]:
# =====================================================================
# PHYSICAL AI DEMO - SMART FACTORY / SMART MANUFACTURING
# Google Colab - SINGLE CELL
#
# Demonstrates:
#   Conveyor belt
#   Moving products
#   Simulated camera / computer vision inspection
#   Defect detection
#   Sensor readings
#   AI decision
#   Physical actuator
#   Accept / Reject operation
#   Feedback loop
#
# EDUCATIONAL SIMULATION
# =====================================================================


# =====================================================================
# 1. INSTALL / IMPORT
# =====================================================================

!pip install -q opencv-python

import cv2
import numpy as np
import random
import os
import math

from IPython.display import HTML, display
from base64 import b64encode


# =====================================================================
# 2. VIDEO SETTINGS
# =====================================================================

WIDTH = 1280
HEIGHT = 720

FPS = 25
DURATION = 20

TOTAL_FRAMES = FPS * DURATION


output_video = "smart_factory_raw.mp4"
final_video = "smart_factory_demo.mp4"


writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (WIDTH, HEIGHT)
)


# =====================================================================
# 3. FACTORY PARAMETERS
# =====================================================================

conveyor_speed = 8.0

accepted_count = 0
rejected_count = 0

machine_temperature = 42.0
machine_vibration = 1.8

inspection_x = 700
reject_x = 930

In [ ]:
# =====================================================================
# 4. PRODUCT CLASS
# =====================================================================

class Product:

    def __init__(self, product_id, x):

        self.id = product_id

        self.x = x

        self.y = 420

        self.width = 100
        self.height = 75

        # Random product type
        self.product_type = random.choice([
            "Part-A",
            "Part-B",
            "Part-C"
        ])

        # 25% chance of defect
        self.defective = (
            random.random() < 0.25
        )

        self.inspected = False

        self.decision = "WAITING"

        self.rejected = False

        self.finished = False

        # Random defect type
        if self.defective:

            self.defect_type = random.choice([
                "Crack",
                "Surface defect",
                "Wrong shape"
            ])

        else:

            self.defect_type = "None"


# =====================================================================
# 5. INITIAL PRODUCTS
# =====================================================================

products = []

next_product_id = 1

for i in range(5):

    products.append(
        Product(
            next_product_id,
            x=80 - i * 250
        )
    )

    next_product_id += 1


# =====================================================================
# 6. DRAW FACTORY BACKGROUND
# =====================================================================

def draw_factory(frame):

    # Background
    frame[:] = (225, 225, 225)

    # -------------------------------------------------------------
    # Factory title
    # -------------------------------------------------------------

    cv2.putText(
        frame,
        "PHYSICAL AI - SMART FACTORY",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (20, 20, 20),
        3
    )


    # -------------------------------------------------------------
    # Conveyor belt
    # -------------------------------------------------------------

    cv2.rectangle(
        frame,
        (40, 450),
        (1190, 560),
        (70, 70, 70),
        -1
    )


    cv2.rectangle(
        frame,
        (40, 455),
        (1190, 540),
        (110, 110, 110),
        -1
    )


    # Conveyor rollers

    for x in range(60, 1180, 70):

        cv2.circle(
            frame,
            (x, 555),
            18,
            (40, 40, 40),
            -1
        )


    # Direction arrow

    cv2.arrowedLine(
        frame,
        (180, 600),
        (430, 600),
        (0, 120, 255),
        6,
        tipLength=0.12
    )


    cv2.putText(
        frame,
        "CONVEYOR DIRECTION",
        (185, 640),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (30, 30, 30),
        2
    )


    # -------------------------------------------------------------
    # Camera inspection station
    # -------------------------------------------------------------

    cv2.rectangle(
        frame,
        (inspection_x - 70, 175),
        (inspection_x + 70, 265),
        (60, 60, 60),
        -1
    )


    cv2.circle(
        frame,
        (inspection_x, 220),
        28,
        (0, 100, 255),
        -1
    )


    cv2.circle(
        frame,
        (inspection_x, 220),
        15,
        (0, 0, 0),
        -1
    )


    cv2.line(
        frame,
        (inspection_x, 265),
        (inspection_x, 420),
        (0, 255, 255),
        3
    )


    cv2.putText(
        frame,
        "AI VISION",
        (inspection_x - 70, 150),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 0),
        2
    )


    # -------------------------------------------------------------
    # Reject actuator
    # -------------------------------------------------------------

    cv2.rectangle(
        frame,
        (reject_x - 30, 320),
        (reject_x + 30, 410),
        (120, 120, 120),
        -1
    )


    cv2.putText(
        frame,
        "REJECT ACTUATOR",
        (reject_x - 100, 295),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 0, 0),
        2
    )


    # Reject bin

    cv2.rectangle(
        frame,
        (1010, 570),
        (1180, 690),
        (80, 80, 180),
        -1
    )


    cv2.putText(
        frame,
        "REJECT BIN",
        (1025, 630),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


# =====================================================================
# 7. DRAW PRODUCT
# =====================================================================

def draw_product(frame, product):

    x = int(product.x)
    y = int(product.y)


    # -------------------------------------------------------------
    # Product color
    # -------------------------------------------------------------

    if not product.inspected:

        color = (180, 140, 60)

    elif product.decision == "ACCEPT":

        color = (0, 180, 0)

    else:

        color = (0, 0, 220)


    # -------------------------------------------------------------
    # Draw product
    # -------------------------------------------------------------

    cv2.rectangle(
        frame,
        (x, y),
        (
            x + product.width,
            y + product.height
        ),
        color,
        -1
    )


    # -------------------------------------------------------------
    # Defect mark
    # -------------------------------------------------------------

    if product.defective:

        cv2.line(
            frame,
            (x + 25, y + 15),
            (x + 70, y + 55),
            (255, 255, 255),
            5
        )


    # -------------------------------------------------------------
    # Bounding box = simulated Computer Vision detection
    # -------------------------------------------------------------

    cv2.rectangle(
        frame,
        (x - 5, y - 5),
        (
            x + product.width + 5,
            y + product.height + 5
        ),
        (0, 255, 255),
        2
    )


    # -------------------------------------------------------------
    # Label
    # -------------------------------------------------------------

    label = (
        f"{product.product_type} "
        f"ID:{product.id}"
    )


    cv2.putText(
        frame,
        label,
        (x, y - 12),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (0, 0, 0),
        2
    )


    if product.inspected:

        cv2.putText(
            frame,
            product.decision,
            (x + 10, y + 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.58,
            (255, 255, 255),
            2
        )


# =====================================================================
# 8. AI / COMPUTER VISION INSPECTION
# =====================================================================

def inspect_product(product):

    """

    Simulates a Computer Vision inspection result.

    In a real factory this function could be replaced by:
        YOLO
        CNN
        anomaly detection model
        industrial vision model

    """

    confidence = random.uniform(
        0.88,
        0.99
    )


    if product.defective:

        decision = "REJECT"

    else:

        decision = "ACCEPT"


    return (
        decision,
        confidence
    )


# =====================================================================
# 9. PHYSICAL AI SAFETY / MACHINE CONTROLLER
# =====================================================================

def factory_controller():

    global machine_temperature
    global machine_vibration
    global conveyor_speed


    # Simulate changing sensor values

    machine_temperature += random.uniform(
        -0.08,
        0.10
    )

    machine_vibration += random.uniform(
        -0.03,
        0.03
    )


    # Safety control

    if machine_temperature > 60:

        conveyor_speed = 3

        return "SLOW LINE - HIGH TEMPERATURE"


    if machine_vibration > 4:

        conveyor_speed = 3

        return "SLOW LINE - HIGH VIBRATION"


    conveyor_speed = 8

    return "NORMAL"


# =====================================================================
# 10. DRAW PHYSICAL AI DASHBOARD
# =====================================================================

def draw_dashboard(
    frame,
    factory_status,
    active_product,
    active_confidence
):


    # Dashboard background

    cv2.rectangle(
        frame,
        (25, 75),
        (510, 340),
        (25, 25, 25),
        -1
    )


    cv2.putText(
        frame,
        "SMART MANUFACTURING CONTROL",
        (45, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.67,
        (0, 255, 0),
        2
    )


    cv2.putText(
        frame,
        f"Conveyor Speed : {conveyor_speed:.1f} m/min",
        (45, 150),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )


    cv2.putText(
        frame,
        f"Temperature    : {machine_temperature:.1f} C",
        (45, 180),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )


    cv2.putText(
        frame,
        f"Vibration      : {machine_vibration:.2f}",
        (45, 210),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )


    cv2.putText(
        frame,
        f"Accepted       : {accepted_count}",
        (45, 240),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 0),
        1
    )


    cv2.putText(
        frame,
        f"Rejected       : {rejected_count}",
        (45, 270),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 100, 255),
        1
    )


    cv2.putText(
        frame,
        f"Machine Status : {factory_status}",
        (45, 305),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.48,
        (0, 255, 255),
        1
    )


    # -------------------------------------------------------------
    # Current AI decision
    # -------------------------------------------------------------

    if active_product is not None:

        cv2.putText(
            frame,
            f"Vision: {active_product.product_type}",
            (760, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (0, 0, 0),
            2
        )


        cv2.putText(
            frame,
            f"Defect: {active_product.defect_type}",
            (760, 135),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (0, 0, 0),
            2
        )


        cv2.putText(
            frame,
            f"Confidence: {active_confidence:.2f}",
            (760, 170),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (0, 0, 0),
            2
        )


        cv2.putText(
            frame,
            f"AI Decision: {active_product.decision}",
            (760, 205),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (
                (0, 180, 0)
                if active_product.decision == "ACCEPT"
                else
                (0, 0, 220)
            ),
            2
        )


# =====================================================================
# 11. MAIN PHYSICAL AI SIMULATION
# =====================================================================

active_product = None
active_confidence = 0.0


for frame_number in range(TOTAL_FRAMES):


    # -------------------------------------------------------------
    # Create image
    # -------------------------------------------------------------

    frame = np.zeros(
        (
            HEIGHT,
            WIDTH,
            3
        ),
        dtype=np.uint8
    )


    draw_factory(frame)


    # -------------------------------------------------------------
    # Machine sensor / controller
    # -------------------------------------------------------------

    factory_status = factory_controller()


    # -------------------------------------------------------------
    # Update each physical product
    # -------------------------------------------------------------

    for product in products:


        if product.finished:

            continue


        # Conveyor movement

        product.x += conveyor_speed * 0.55


        # =========================================================
        # CAMERA / AI INSPECTION
        # =========================================================

        if (
            not product.inspected
            and
            product.x >= inspection_x - 45
        ):


            decision, confidence = inspect_product(
                product
            )


            product.decision = decision

            product.inspected = True

            active_product = product

            active_confidence = confidence


            print(
                f"Product {product.id}: "
                f"{product.product_type} | "
                f"Defect={product.defect_type} | "
                f"AI={decision} | "
                f"Confidence={confidence:.2f}"
            )


        # =========================================================
        # PHYSICAL ACTUATOR
        # =========================================================

        if (
            product.inspected
            and
            product.decision == "REJECT"
            and
            not product.rejected
            and
            product.x >= reject_x
        ):


            product.rejected = True


            # Simulated pneumatic actuator pushes product down

            product.y += 150


            rejected_count += 1


        # ---------------------------------------------------------
        # Accepted product
        # ---------------------------------------------------------

        if (
            product.inspected
            and
            product.decision == "ACCEPT"
            and
            product.x > WIDTH
            and
            not product.finished
        ):


            accepted_count += 1

            product.finished = True


        # ---------------------------------------------------------
        # Rejected product reaches bin
        # ---------------------------------------------------------

        if (
            product.rejected
            and
            product.x > 1120
        ):

            product.finished = True


        draw_product(
            frame,
            product
        )


    # =============================================================
    # CREATE NEW PRODUCTS
    # =============================================================

    active_products = [
        p
        for p in products
        if not p.finished
    ]


    if active_products:

        minimum_x = min(
            p.x
            for p in active_products
        )

    else:

        minimum_x = WIDTH


    # Add new product every few seconds

    if (
        frame_number % 90 == 0
        and
        frame_number > 0
    ):


        new_product = Product(
            next_product_id,
            -130
        )


        products.append(
            new_product
        )


        next_product_id += 1


    # =============================================================
    # DRAW REJECT ACTUATOR WHEN ACTIVE
    # =============================================================

    reject_active = any(

        p.rejected
        and
        reject_x - 60 < p.x < reject_x + 140

        for p in products

    )


    if reject_active:


        cv2.arrowedLine(
            frame,
            (reject_x, 390),
            (reject_x + 80, 480),
            (0, 0, 255),
            10,
            tipLength=0.25
        )


        cv2.putText(
            frame,
            "ACTUATOR ACTIVE",
            (reject_x - 90, 375),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 0, 255),
            2
        )


    # =============================================================
    # DASHBOARD
    # =============================================================

    draw_dashboard(
        frame,
        factory_status,
        active_product,
        active_confidence
    )


    # -------------------------------------------------------------
    # Physical AI loop description
    # -------------------------------------------------------------

    cv2.putText(
        frame,
        "Sense -> Perceive -> Decide -> Act -> Feedback",
        (400, 690),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (20, 20, 20),
        2
    )


    writer.write(
        frame
    )


# =====================================================================
# 12. CLOSE VIDEO
# =====================================================================

writer.release()


# =====================================================================
# 13. CONVERT VIDEO TO H264
# =====================================================================

os.system(

    f'ffmpeg -y '
    f'-loglevel error '
    f'-i "{output_video}" '
    f'-vcodec libx264 '
    f'-pix_fmt yuv420p '
    f'"{final_video}"'

)


# =====================================================================
# 14. DISPLAY VIDEO DIRECTLY IN GOOGLE COLAB
# =====================================================================

with open(
    final_video,
    "rb"
) as f:

    video_bytes = f.read()


encoded_video = b64encode(
    video_bytes
).decode()


print()
print("==========================================================")
print("SMART FACTORY PHYSICAL AI SIMULATION")
print("==========================================================")


display(

    HTML(

        f'''

        <video
            width="1000"
            controls
            autoplay
            loop>

            <source
                src="data:video/mp4;base64,{encoded_video}"
                type="video/mp4">

        </video>

        '''

    )

)